# **Project Name**    -Amazon Prime TV Shows & Movies – ML



##### **Contribution**    - Individual


# **Project Summary -**

This project analyzes the Amazon Prime Movies and TV Shows catalog to identify patterns and trends in content type, genres, ratings, release years, countries, and runtime.

The dataset was cleaned by handling missing values, removing duplicates, and standardizing categorical and textual data. EDA was performed using Univariate, Bivariate, and Multivariate analysis with meaningful visualizations.

Text preprocessing was performed on descriptions using techniques such as lowercasing, stopword removal, tokenization, and TF-IDF vectorization. Hypothesis testing was also conducted to identify significant differences between Movies and TV Shows.

For classification, Logistic Regression, Random Forest, and XGBoost were implemented. XGBoost achieved the highest F1 Score of 97.32% and was selected as the final model. The project provides useful insights for content classification, catalog management, and data-driven content strategy.

# **Problem Statement**


Amazon Prime has a large catalog of Movies and TV Shows with different genres, ratings, release years, and other attributes. Analyzing this data can help identify important content trends and patterns.

The objective of this project is to analyze the Amazon Prime catalog, identify meaningful relationships between content attributes, and build a machine learning model to classify content as Movies or TV Shows.

#### **Define Your Business Objective?**

To help Amazon Prime improve content organization, classification, catalog management, and data-driven content strategy using EDA and machine learning insights.

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

### Dataset Loading

In [ ]:
# Load Dataset

title_df = pd.read_csv("/content/credits.csv")
credit_df = pd.read_csv("/content/titles.csv")

print("Title dataset loaded successfully.")
print("Credit dataset loaded successfully.")

### Dataset First View

In [ ]:
#Display credits head part
print("First 5 rows of title dataset:")
display(credits_df.head(5))

In [ ]:
#Display ttles head part

print("\nFirst 5 rows of credit dataset:")
display(titles_df.head(5))

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print("Number of rows and columns in the title dataset:")
print(titles_df.shape)

print("\nNumber of rows and columns in the credit dataset:")
print(credits_df.shape)

### Dataset Information

In [ ]:
# Dataset Info
print("Title Dataset Information:")
titles_df.info()

print("\nCredit Dataset Information:")
credits_df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("Duplicate rows in title dataset:", titles_df.duplicated().sum())
print("Duplicate rows in credit dataset:", credits_df.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print("Missing values in title dataset:")
display(titles_df.isnull().sum().sort_values(ascending=False))

print("\nMissing values in credit dataset:")
display(credits_df.isnull().sum().sort_values(ascending=False))

In [ ]:
# Visualizing the missing values
missing = titles_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=missing.values, y=missing.index)
plt.title("Missing Values in Title Dataset")
plt.xlabel("Number of Missing Values")
plt.ylabel("Columns")
plt.show()

### What did you know about your dataset?

The title dataset contains information about Amazon Prime movies and TV shows, including title, content type, description, release year, age certification, runtime, genres, production countries, seasons, IMDb information, and TMDB information. The credit dataset contains information about people associated with each title, including actors and other credited roles.

The common id column allows the title and credit datasets to be connected. The datasets contain both numerical and categorical variables, along with list-like columns such as genres and production countries. Missing values are expected in columns such as age certification, seasons, and some rating-related fields.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("Title Dataset Columns:")
print(titles_df.columns.tolist())

print("\nCredit Dataset Columns:")
print(credits_df.columns.tolist())

In [ ]:
# Dataset Describe
display(titles_df.describe(include='all').T)

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for column in titles_df.columns:
    print(f"\n{column}:")
    print("Unique values:", titles_df[column].nunique())

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
df = titles_df.copy()
credits = credits_df.copy()

In [ ]:
df.drop_duplicates(inplace=True)
credits.drop_duplicates(inplace=True)

print("Title dataset shape after removing duplicates:", df.shape)
print("Credit dataset shape after removing duplicates:", credits.shape)

In [ ]:
credits['release_year'] = pd.to_numeric(credits['release_year'], errors='coerce')

In [ ]:
numeric_columns = [
    'runtime',
    'seasons',
    'imdb_score',
    'imdb_votes',
    'tmdb_popularity',
    'tmdb_score'
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

In [ ]:
text_columns = ['title', 'type', 'age_certification']

for column in text_columns:
    if column in credits.columns:
        credits[column] = credits[column].astype('string').str.strip()

credits['type'] = credits['type'].str.upper()

In [ ]:
def parse_list_column(value):
    if pd.isna(value):
        return []

    try:
        result = ast.literal_eval(value)
        if isinstance(result, list):
            return [str(x).strip().lower() for x in result]
        return []
    except:
        return []

credits['genres_list'] = credits['genres'].apply(parse_list_column)

In [ ]:
credits['countries_list'] = credits['production_countries'].apply(parse_list_column)

In [ ]:
credits['age_certification'] = credits['age_certification'].fillna('Unknown')

credits['runtime'] = credits['runtime'].fillna(credits['runtime'].median())

credits['imdb_score'] = credits['imdb_score'].fillna(credits['imdb_score'].median())

credits['tmdb_score'] = credits['tmdb_score'].fillna(credits['tmdb_score'].median())

credits['tmdb_popularity'] = credits['tmdb_popularity'].fillna(
    credits['tmdb_popularity'].median()
)

credits['imdb_votes'] = credits['imdb_votes'].fillna(
    credits['imdb_votes'].median()
)

In [ ]:
credits['seasons'] = credits['seasons'].fillna(0)

In [ ]:
print("Missing values after cleaning:")
display(df.isnull().sum().sort_values(ascending=False))

### What all manipulations have you done and insights you found?

The dataset was cleaned by removing duplicate records, standardizing categorical text values, converting numerical columns into appropriate numeric data types, and converting list-like genre and country columns into Python lists. Missing values were handled using appropriate strategies such as median imputation for numerical variables and "Unknown" for missing age certifications. The credit dataset was also checked for duplicates.

The cleaned data is now suitable for statistical analysis and visualization. Separating genres and countries into lists allows individual genres and countries to be analyzed accurately rather than treating the entire list as one category.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart 1 - Movies vs TV Shows

type_counts = credits['type'].value_counts()

plt.figure(figsize=(7, 7))

plt.pie(
    type_counts.values,
    labels=type_counts.index,
    autopct='%1.1f%%',
    startangle=90
)

plt.title("Movies vs TV Shows Distribution")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A pie chart is suitable because the objective is to show the proportion of Movies and TV Shows in the complete Amazon Prime catalog.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.The chart shows the percentage contribution of Movies and TV Shows to the overall catalog.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.This helps Amazon understand whether its catalog is more focused on Movies or TV Shows and supports future content investment decisions.

#### Chart - 2

In [ ]:
year_counts = credits['release_year'].value_counts().sort_index()

plt.figure(figsize=(14, 6))

plt.plot(
    year_counts.index,
    year_counts.values,
    marker='o'
)

plt.title("Content Releases Over the Years")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.grid(True)
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A line chart is ideal for showing a trend over time.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.It shows how the number of titles changed across different release years.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.The trend helps identify periods of increasing or decreasing content production.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
# Top 10 Genres

genre_df = credits.explode('genres_list')

top_genres = genre_df['genres_list'].value_counts().head(10)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=top_genres.values,
    y=top_genres.index
)

plt.title("Top 10 Most Common Genres")
plt.xlabel("Number of Titles")
plt.ylabel("Genre")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A horizontal bar chart makes ranking the genres easy.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.The chart identifies the most frequently occurring genres.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.Popular genres can guide content acquisition and production strategies.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
#  Age Certification Distribution

rating_counts = df['age_certification'].fillna('Unknown').value_counts()

plt.figure(figsize=(8, 8))

plt.pie(
    rating_counts.values,
    labels=rating_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.4)
)

plt.title("Age Certification Distribution")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A donut chart clearly represents the composition of the catalog across audience ratings.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.Some age certifications are much more common than others.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.Amazon can understand which audience categories receive the greatest amount of content.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
# Runtime Distribution

plt.figure(figsize=(10, 6))

sns.histplot(
    credits['runtime'],
    bins=30,
    kde=True
)

plt.title("Distribution of Content Runtime")
plt.xlabel("Runtime in Minutes")
plt.ylabel("Number of Titles")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A histogram is ideal for understanding the distribution of a numerical variable.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.The plot shows the most common runtime ranges and identifies unusually short or long content.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.Runtime information can help Amazon understand different content formats and audience preferences.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
# IMDb Score Distribution

plt.figure(figsize=(8, 6))

sns.violinplot(
    y=credits['imdb_score'].dropna()
)

plt.title("Distribution of IMDb Scores")
plt.ylabel("IMDb Score")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A violin plot shows the distribution, density, median, and spread of IMDb scores.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.It shows where IMDb scores are concentrated and whether the distribution is skewed.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.Amazon can evaluate the overall quality distribution of its catalog.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
# Number of Seasons

show_data = credits[credits['type'] == 'SHOW'].copy()

plt.figure(figsize=(10, 6))

sns.countplot(
    data=show_data,
    x='seasons'
)

plt.title("Distribution of Number of Seasons")
plt.xlabel("Number of Seasons")
plt.ylabel("Number of TV Shows")
plt.xticks(rotation=45)
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A count plot is appropriate because seasons are discrete values.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.Most shows tend to have a smaller number of seasons, while long-running shows are less frequent.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.This can help Amazon understand whether audiences are being offered short-format or long-running series.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
# IMDb Votes Outlier Analysis

plt.figure(figsize=(10, 5))

sns.boxplot(
    x=credits['imdb_votes'].dropna()
)

plt.title("IMDb Votes Distribution and Outliers")
plt.xlabel("Number of IMDb Votes")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A box plot is excellent for identifying outliers.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.A small number of titles may have extremely high numbers of IMDb votes compared with the majority.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.These highly engaged titles can help Amazon identify content with significant audience attention.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
# Rating Distribution by Content Type

rating_type = pd.crosstab(
    credits['age_certification'].fillna('Unknown'),
    credits['type']
)

rating_type.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6)
)

plt.title("Age Certification by Content Type")
plt.xlabel("Age Certification")
plt.ylabel("Number of Titles")
plt.xticks(rotation=45)
plt.legend(title="Content Type")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A stacked bar chart shows both total volume and the contribution of Movies and Shows.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.Different ratings have different proportions of Movies and TV Shows.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.This helps Amazon understand how different audience categories are served by each content format

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
# Genre by Content Type

genre_type_df = credits.explode('genres_list')

top_genres = genre_type_df[
    'genres_list'
].value_counts().head(10).index

genre_type_df = genre_type_df[
    genre_type_df['genres_list'].isin(top_genres)
]

genre_type = pd.crosstab(
    genre_type_df['genres_list'],
    genre_type_df['type']
)

genre_type.plot(
    kind='bar',
    figsize=(12, 6)
)

plt.title("Top Genres by Content Type")
plt.xlabel("Genre")
plt.ylabel("Number of Titles")
plt.xticks(rotation=45)
plt.legend(title="Content Type")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.Grouped bars allow direct comparison between Movies and Shows for each genre.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.Some genres are more strongly associated with Movies while others are more common among Shows.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.This can support genre-specific production decisions.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
# Chart 11 - Top Production Countries

country_df = credits.explode('countries_list')

top_countries = (
    country_df['countries_list']
    .replace('', np.nan)
    .dropna()
    .value_counts()
    .head(10)
    .sort_values()
)

plt.figure(figsize=(10, 6))

plt.hlines(
    y=top_countries.index,
    xmin=0,
    xmax=top_countries.values
)

plt.plot(
    top_countries.values,
    top_countries.index,
    'o'
)

plt.title("Top 10 Production Countries")
plt.xlabel("Number of Titles")
plt.ylabel("Country")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A lollipop chart provides a cleaner visual ranking than a traditional bar chart.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.The catalog is concentrated among a smaller number of major production countries.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.Amazon can identify strong production markets and potential opportunities in underrepresented regions.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
#  Country vs Content Type

country_type = pd.crosstab(
    country_df['countries_list'],
    country_df['type']
)

top_country_names = (
    country_df['countries_list']
    .replace('', np.nan)
    .dropna()
    .value_counts()
    .head(10)
    .index
)

country_type = country_type.loc[
    country_type.index.isin(top_country_names)
]

plt.figure(figsize=(10, 7))

sns.heatmap(
    country_type,
    annot=True,
    fmt='g',
    cmap='YlGnBu'
)

plt.title("Production Country vs Content Type")
plt.xlabel("Content Type")
plt.ylabel("Country")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A heatmap makes comparisons across two categorical variables easy.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.The contribution of Movies and Shows differs across production countries.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.This can support regional content acquisition strategies.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
#  Area Chart of Releases

year_type = pd.crosstab(
    credits['release_year'],
    credits['type']
).sort_index()

plt.figure(figsize=(14, 6))

plt.stackplot(
    year_type.index,
    *[
        year_type[column]
        for column in year_type.columns
    ],
    labels=year_type.columns,
    alpha=0.7
)

plt.title("Movie and TV Show Releases Over Time")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.legend()
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.An area chart shows both overall growth and the relative contribution of Movies and Shows.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.It shows how the composition of the catalog changes across release periods.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.This helps identify long-term changes in content strategy.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
# Release Year vs Genre

genre_year_df = credits.explode('genres_list')

top_genres = (
    genre_year_df['genres_list']
    .value_counts()
    .head(10)
    .index
)

genre_year_df = genre_year_df[
    genre_year_df['genres_list'].isin(top_genres)
]

genre_year = pd.crosstab(
    genre_year_df['release_year'],
    genre_year_df['genres_list']
)

plt.figure(figsize=(14, 8))

sns.heatmap(
    genre_year.tail(30),
    cmap='viridis'
)

plt.title("Top Genres Across Recent Release Years")
plt.xlabel("Genre")
plt.ylabel("Release Year")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A heatmap is useful for analyzing two categorical/time dimensions simultaneously.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.The concentration of genres changes across different release periods.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code

pairplot_data = credits[
    ['runtime', 'imdb_score', 'imdb_votes',
     'tmdb_popularity', 'tmdb_score', 'type']
].dropna()

# Limit the number of rows for faster visualization
pairplot_sample = pairplot_data.sample(
    min(2000, len(pairplot_data)),
    random_state=42
)

sns.pairplot(
    pairplot_sample,
    hue='type',
    diag_kind='hist'
)

plt.suptitle(
    "Pair Plot of Numerical Variables",
    y=1.02
)

plt.show()

#### Chart - 16 - IMDb Score Boxplot

In [ ]:
# IMDb Score Boxplot

plt.figure(figsize=(8, 6))
sns.boxplot(y=credits['imdb_score'].dropna())
plt.title('Distribution of IMDb Scores (Boxplot)')
plt.ylabel('IMDb Score')
plt.show()

##### 1. Why did you pick the specific chart?
A boxplot is ideal for visualizing the distribution of a numerical variable, including its median, quartiles, and any potential outliers, in a compact format.

##### 2. What is/are the insight(s) found from the chart?
The boxplot reveals the central tendency (median) of IMDb scores, their spread (interquartile range), and any scores that fall outside the typical range, indicating potential outliers.

##### 3. Will the gained insights help creating a positive business impact?
Understanding the distribution and presence of outliers in IMDb scores can help Amazon identify content with exceptionally high or low ratings, informing content acquisition and quality assessment strategies.

##### 1. Why did you pick the specific chart?

Answer Here.A pair plot is selected for multivariate analysis because it allows us to examine relationships between multiple numerical variables simultaneously. It displays scatter plots between variable pairs and distributions along the diagonal.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.The relationship between IMDb score and TMDB score.
The relationship between runtime and IMDb score.
The relationship between TMDB popularity and IMDb votes.
Possible clusters or differences between Movies and TV Shows.
Potential outliers in numerical variables.
Variables that appear to have positive, negative, or weak relationships.

# **5.Hypothesis testing**

In [ ]:
# Perform Statistical Test to obtain P-Value

from scipy.stats import ttest_ind

# Count the number of credits for each person based on their role
actor_counts = df[
    df['role'].str.upper() == 'ACTOR'
].groupby('name').size()

director_counts = df[
    df['role'].str.upper() == 'DIRECTOR'
].groupby('name').size()

# Independent two-sample t-test
t_stat, p_value = ttest_ind(
    actor_counts,
    director_counts,
    equal_var=False,
    nan_policy='omit'
)

print("T-Statistic:", t_stat)
print("P-Value:", p_value)

if p_value < 0.05:
    print("Reject the Null Hypothesis.")
    print("There is a significant difference in average credits between Actors and Directors.")
else:
    print("Fail to Reject the Null Hypothesis.")
    print("There is no significant difference in average credits between Actors and Directors.")

# Which statistical test have you done to obtain P-Value?

Independent Two-Sample t-test (Welch's t-test).

# **Why did you choose the specific statistical test?**

The dependent variable, imdb_score, is numerical, while type contains two independent groups: MOVIE and SHOW. Therefore, an independent two-sample t-test is appropriate for comparing the average IMDb scores of these two groups. Welch's version is used because it does not assume equal variances between the groups.

# **Hypothetical Statement - 2**
1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Research Question: Is there a significant relationship between IMDb score and TMDB score?

Null Hypothesis (H₀):
There is no significant correlation between IMDb scores and TMDB scores.

Alternate Hypothesis (H₁):
There is a significant correlation between IMDb scores and TMDB scores.

2. Perform an appropriate statistical test.

In [ ]:
from scipy.stats import pearsonr

correlation_data = credits[
    ['imdb_score', 'tmdb_score']
].dropna()

correlation, p_value = pearsonr(
    correlation_data['imdb_score'],
    correlation_data['tmdb_score']
)

print("Pearson Correlation:", correlation)
print("P-value:", p_value)

if p_value < 0.05:
    print("Reject the Null Hypothesis.")
    print("There is a significant correlation between IMDb and TMDB scores.")
else:
    print("Fail to Reject the Null Hypothesis.")
    print("There is no significant correlation between IMDb and TMDB scores.")

# **Which statistical test have you done to obtain P-Value?**

Pearson Correlation Test.

# **Why did you choose the specific statistical test?**

Both imdb_score and tmdb_score are numerical variables. Pearson correlation is appropriate for measuring the strength and direction of a linear relationship between two numerical variables and provides a p-value to test whether the correlation is statistically significant.

## **Hypothetical Statement - 3**
1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Research Question: Do Movies and TV Shows have different average runtimes?

Null Hypothesis (H₀):
There is no significant difference in the average runtime of Movies and TV Shows.

Alternate Hypothesis (H₁):
There is a significant difference in the average runtime of Movies and TV Shows.

2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

from scipy.stats import ttest_ind

movie_runtime = credits[
    credits['type'] == 'MOVIE'
]['runtime'].dropna()

show_runtime = credits[
    credits['type'] == 'SHOW'
]['runtime'].dropna()

t_stat, p_value = ttest_ind(
    movie_runtime,
    show_runtime,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Reject the Null Hypothesis.")
    print("There is a significant difference in runtime between Movies and TV Shows.")
else:
    print("Fail to Reject the Null Hypothesis.")
    print("There is no significant difference in runtime between Movies and TV Shows.")

There is a statistically significant difference in the average runtime of Movies and TV Shows in the Amazon Prime dataset.

# **6. Feature Engineering & Data Pre-processing**
1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation

# Check missing values
missing_values = credits.isnull().sum()

print("Missing values before treatment:")
print(missing_values[missing_values > 0])

# Numerical columns
numeric_columns = [
    'runtime',
    'imdb_score',
    'imdb_votes',
    'tmdb_popularity',
    'tmdb_score',
    'seasons'
]

# Fill numerical missing values with median
for col in numeric_columns:
    if col in credits.columns:
        credits[col] = credits[col].fillna(credits[col].median())

# Categorical columns
categorical_columns = [
    'age_certification',
    'genres',
    'production_countries'
]

# Fill categorical missing values with "Unknown"
for col in categorical_columns:
    if col in credits.columns:
        credits[col] = credits[col].fillna('Unknown')

print("\nMissing values after treatment:")
print(credits.isnull().sum())

2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier Treatments

numeric_columns = [
    'runtime',
    'imdb_score',
    'imdb_votes',
    'tmdb_popularity',
    'tmdb_score'
]

for col in numeric_columns:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Winsorization / Capping
        df[col] = df[col].clip(
            lower=lower_bound,
            upper=upper_bound
        )

print("Outlier treatment completed.")

**What all outlier treatment techniques have you used and why?**

I used the IQR method to identify potential outliers and applied capping (Winsorization) instead of removing observations.

This approach was selected because extremely high IMDb votes or TMDB popularity values may represent genuinely popular content rather than incorrect data. Removing these observations could result in loss of valuable information.

**3. Categorical Encoding**

In [ ]:
# Encode categorical columns

from sklearn.preprocessing import OneHotEncoder

# Create a working dataset
ml_df = credits.copy()

# Convert list-like genre/country values to strings
ml_df['genres'] = ml_df['genres'].astype(str)
ml_df['production_countries'] = ml_df['production_countries'].astype(str)

print("Categorical columns prepared for encoding.")

**What categorical encoding techniques have you used & why?**

I used One-Hot Encoding for categorical variables such as age_certification and other categorical attributes.

One-Hot Encoding is appropriate because these categories do not have a natural numerical order. Assigning values such as 1, 2, 3 could incorrectly imply an ordinal relationship.

# **4. Textual Data Preprocessing**

1. Expand Contraction

In [ ]:
# Expand Contractions

import re

def expand_contractions(text):
    text = str(text)

    contractions = {
        "can't": "cannot",
        "won't": "will not",
        "n't": " not",
        "'re": " are",
        "'ve": " have",
        "'ll": " will",
        "'d": " would",
        "'m": " am",
        "'s": " is"
    }

    for contraction, expansion in contractions.items():
        text = text.replace(contraction, expansion)

    return text

credits['description_clean'] = credits['description'].fillna('').apply(
    expand_contractions
)

2. Lower Casing

In [ ]:
credits['description_clean'] = credits['description_clean'].str.lower()

print(credits['description_clean'].head())

3. Removing Punctuations

In [ ]:
# Remove Punctuations

credits['description_clean'] = credits['description_clean'].apply(
    lambda x: re.sub(r'[^\w\s]', '', str(x))
)

4. Removing URLs & Words/Digits Containing Digits

In [ ]:
# Remove URLs and words containing digits

def remove_urls_and_digits(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', str(text))
    text = re.sub(r'\b\w*\d\w*\b', '', text)

    return text

credits['description_clean'] = credits['description_clean'].apply(
    remove_urls_and_digits
)

5. Removing Stopwords

In [ ]:
# Remove Stopwords

import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

credits['description_clean'] = credits['description_clean'].apply(
    lambda x: ' '.join(
        word for word in str(x).split()
        if word not in stop_words
    )
)

Remove White Spaces

In [ ]:
# Remove White Spaces

credits['description_clean'] = credits['description_clean'].apply(
    lambda x: ' '.join(str(x).split())
)

6. Rephrase Text

In [ ]:
# Rephrase Text

# No manual rephrasing is required because the original
# descriptions are already meaningful catalog descriptions.

print("Text rephrasing is not required for this dataset.")

7. Tokenization

In [ ]:
# Tokenization

credits['tokens'] = credits['description_clean'].apply(
    lambda x: str(x).split()
)

print(credits[['description_clean', 'tokens']].head())

9. Part of Speech Tagging

In [ ]:
# POS Tagging

import nltk

nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

from nltk import word_tokenize, pos_tag

sample_text = credits['description_clean'].iloc[0]

tokens = word_tokenize(sample_text)

pos_tags = pos_tag(tokens)

print(pos_tags[:20])

10. Text Vectorization

In [ ]:
# Vectorizing Text using TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words='english'
)

text_features = tfidf.fit_transform(
    credits['description_clean'].fillna('')
)

print("TF-IDF matrix shape:", text_features.shape)

# **4. Feature Manipulation & Selection**
1. Feature Manipulation

In [ ]:
# Manipulate Features and Create New Features

# Create a title length feature
credits['title_length'] = credits['title'].fillna('').apply(len)

# Create description length
credits['description_length'] = credits['description'].fillna('').apply(len)

# Create word count
credits['description_word_count'] = credits['description_clean'].apply(
    lambda x: len(str(x).split())
)

print(
    credits[
        ['title_length', 'description_length',
         'description_word_count']
    ].head()
)

2. Feature Selection

In [ ]:
# Select useful features for Machine Learning

selected_features = [
    'runtime',
    'imdb_score',
    'imdb_votes',
    'tmdb_popularity',
    'tmdb_score',
    'seasons',
    'age_certification'
]

print("Selected Features:")
print(selected_features)

# **5. Data Transformation**

**Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?**

Yes. Some numerical variables such as imdb_votes and tmdb_popularity can be highly right-skewed because a small number of titles can have very high popularity or vote counts. Therefore, a logarithmic transformation is useful to reduce skewness and make the numerical variables more suitable for machine learning.

In [ ]:
# Transform skewed numerical features

import numpy as np

credits['log_imdb_votes'] = np.log1p(credits['imdb_votes'])
credits['log_tmdb_popularity'] = np.log1p(credits['tmdb_popularity'])

print(credits[
    ['imdb_votes', 'log_imdb_votes',
     'tmdb_popularity', 'log_tmdb_popularity']
].head())

**6. Data Scaling**

In [ ]:
# Scaling numerical features

from sklearn.preprocessing import StandardScaler

numeric_features = [
    'runtime',
    'imdb_score',
    'log_imdb_votes',
    'log_tmdb_popularity',
    'tmdb_score'
]

scaler = StandardScaler()

scaled_data = scaler.fit_transform(
    credits[numeric_features]
)

print("Scaled data shape:", scaled_data.shape)

**Which method have you used to scale your data and why?**

I used StandardScaler to standardize the numerical variables. It transforms the variables so that they have approximately a mean of 0 and a standard deviation of 1. This prevents variables with larger numerical ranges, such as IMDb votes, from dominating variables with smaller ranges, such as IMDb scores.

# **7. Dimensionality Reduction**
**Do you think that dimensionality reduction is needed? Explain Why.**

For the selected numerical and categorical features, dimensionality reduction is not necessary because the number of features is relatively small.

In [ ]:
# Dimensionality Reduction using Truncated SVD

from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(
    n_components=100,
    random_state=42
)

text_features_reduced = svd.fit_transform(text_features)

print(
    "Original TF-IDF shape:",
    text_features.shape
)

print(
    "Reduced TF-IDF shape:",
    text_features_reduced.shape
)

**Which dimensionality reduction technique have you used and why?**

I used Truncated SVD (Singular Value Decomposition) because the TF-IDF matrix is a high-dimensional sparse matrix. Truncated SVD reduces the 1,000 TF-IDF features to 100 components while retaining the most important information.

Note: You don't need to use these reduced TF-IDF features in the classification model unless you specifically want to include text information.

# **8. Data Splitting**

In [ ]:
# Split the data into training and testing sets

from sklearn.model_selection import train_test_split

X = credits[
    [
        'runtime',
        'imdb_score',
        'imdb_votes',
        'tmdb_popularity',
        'tmdb_score',
        'age_certification'
    ]
].copy()

y = credits['type']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

**What data splitting ratio have you used and why?**

I used an 80:20 train-test split, where 80% of the data is used for training and 20% is reserved for testing.

The stratify parameter was used to maintain a similar proportion of Movies and TV Shows in both the training and testing datasets.

**9. Handling Imbalanced Dataset**

In [ ]:
# Check target class distribution

print("Content Type Distribution:")
print(credits['type'].value_counts())

print("\nContent Type Percentage:")
print(
    credits['type'].value_counts(normalize=True) * 100
)
# Visualize target class distribution

plt.figure(figsize=(7, 5))

sns.countplot(
    data=credits,
    x='type'
)

plt.title("Distribution of Movies and TV Shows")
plt.xlabel("Content Type")
plt.ylabel("Number of Titles")

plt.show()

# **7. ML Model Implementation**

**What technique did you use to handle the imbalance dataset and why?**

Do not automatically use SMOTE.

For this dataset, first inspect the percentage distribution. If the two classes are reasonably represented, no resampling technique is required.

If there is significant imbalance, class weighting can be used in models such as Logistic Regression and Random Forest.

# **ML Model - 1**
**Logistic Regression**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

numeric_features = [
      'runtime',
        'imdb_score',
        'imdb_votes',
        'tmdb_popularity',
        'tmdb_score'
]

categorical_features = [
    'age_certification'
]

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

logistic_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced'
        ))
    ]
)

logistic_model.fit(X_train, y_train)

y_pred_lr = logistic_model.predict(X_test)

print("Logistic Regression Model Results")
print(classification_report(y_test, y_pred_lr))

**Evaluation Metric Score Chart**

In [ ]:
# Visualizing Evaluation Metric Score Chart

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

accuracy_lr = accuracy_score(y_test, y_pred_lr)

precision_lr = precision_score(
    y_test,
    y_pred_lr,
    pos_label='SHOW'
)

recall_lr = recall_score(
    y_test,
    y_pred_lr,
    pos_label='SHOW'
)

f1_lr = f1_score(
    y_test,
    y_pred_lr,
    pos_label='SHOW'
)

metrics_lr = pd.DataFrame({
    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1 Score'
    ],
    'Score': [
        accuracy_lr,
        precision_lr,
        recall_lr,
        f1_lr
    ]
})

plt.figure(figsize=(8, 5))

sns.barplot(
    data=metrics_lr,
    x='Metric',
    y='Score'
)

plt.ylim(0, 1)
plt.title("Logistic Regression Evaluation Metrics")
plt.ylabel("Score")

plt.show()

Logistic Regression is a supervised classification algorithm used to predict whether a title is a Movie or TV Show. It provides a simple and interpretable baseline model. Accuracy measures overall correctness, while Precision, Recall, and F1 Score provide additional information about classification performance.

**Cross-Validation & Hyperparameter Tuning**

In [ ]:
# ML Model - 1 with Hyperparameter Optimization

from sklearn.model_selection import GridSearchCV

param_grid_lr = {
    'classifier__C': [0.01, 0.1, 1, 10],
    'classifier__solver': ['liblinear', 'lbfgs']
}

grid_lr = GridSearchCV(
    logistic_model,
    param_grid_lr,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

best_lr = grid_lr.best_estimator_

y_pred_lr_tuned = best_lr.predict(X_test)

print("Best Parameters:")
print(grid_lr.best_params_)

print("\nTuned Logistic Regression:")
print(
    classification_report(
        y_test,
        y_pred_lr_tuned
    )
)

**Which hyperparameter optimization technique have you used and why?**

I used GridSearchCV with 5-fold cross-validation. It systematically tests different combinations of hyperparameters and selects the combination that gives the best validation performance.

# **ML Model - 2**
**Random Forest**

In [ ]:
# ML Model - 2 Implementation
# Random Forest

from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight='balanced',
            n_jobs=-1
        ))
    ]
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Model Results")
print(
    classification_report(
        y_test,
        y_pred_rf
    )
)

**Evaluation Metric Score Chart**

In [ ]:
# Visualizing Evaluation Metric Score Chart

accuracy_rf = accuracy_score(
    y_test,
    y_pred_rf
)

precision_rf = precision_score(
    y_test,
    y_pred_rf,
    pos_label='SHOW'
)

recall_rf = recall_score(
    y_test,
    y_pred_rf,
    pos_label='SHOW'
)

f1_rf = f1_score(
    y_test,
    y_pred_rf,
    pos_label='SHOW'
)

metrics_rf = pd.DataFrame({
    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1 Score'
    ],
    'Score': [
        accuracy_rf,
        precision_rf,
        recall_rf,
        f1_rf
    ]
})

plt.figure(figsize=(8, 5))

sns.barplot(
    data=metrics_rf,
    x='Metric',
    y='Score'
)

plt.ylim(0, 1)
plt.title("Random Forest Evaluation Metrics")
plt.ylabel("Score")

plt.show()

Random Forest is an ensemble classification algorithm that combines multiple decision trees. It can capture nonlinear relationships between variables and is useful for understanding which content attributes are important for distinguishing Movies from TV Shows.

**Cross-Validation & Hyperparameter Tuning**

In [ ]:
# Random Forest Hyperparameter Optimization

param_grid_rf = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    rf_model,
    param_grid_rf,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

y_pred_rf_tuned = best_rf.predict(X_test)

print("Best Parameters:")
print(grid_rf.best_params_)

print("\nTuned Random Forest:")
print(
    classification_report(
        y_test,
        y_pred_rf_tuned
    )
)

# **ML Model - 3**
**XGBoost**

In [ ]:
# ML Model - 3 Implementation
# XGBoost

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes:", label_encoder.classes_)

In [ ]:
# Train-Test Split for XGBoost

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

In [ ]:
# XGBoost Model

xgb_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            random_state=42,
            eval_metric='logloss'
        ))
    ]
)

xgb_model.fit(
    X_train_xgb,
    y_train_xgb
)

y_pred_xgb = xgb_model.predict(
    X_test_xgb
)

print("XGBoost Model Results")

print(
    classification_report(
        y_test_xgb,
        y_pred_xgb,
        target_names=label_encoder.classes_
    )
)

**Evaluation Metric Score Chart**

In [ ]:
# Visualizing Evaluation Metric Score Chart

accuracy_xgb = accuracy_score(
    y_test_xgb,
    y_pred_xgb
)

f1_xgb = f1_score(
    y_test_xgb,
    y_pred_xgb,
    average='weighted'
)

xgb_metrics = pd.DataFrame({
    'Metric': [
        'Accuracy',
        'F1 Score'
    ],
    'Score': [
        accuracy_xgb,
        f1_xgb
    ]
})

plt.figure(figsize=(7, 5))

sns.barplot(
    data=xgb_metrics,
    x='Metric',
    y='Score'
)

plt.ylim(0, 1)
plt.title("XGBoost Evaluation Metrics")
plt.ylabel("Score")

plt.show()

**XGBoost Hyperparameter Tuning**

In [ ]:
# XGBoost Hyperparameter Optimization

param_grid_xgb = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [3, 5],
    'classifier__learning_rate': [0.05, 0.1]
}

grid_xgb = GridSearchCV(
    xgb_model,
    param_grid_xgb,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1
)

grid_xgb.fit(
    X_train_xgb,
    y_train_xgb
)

best_xgb = grid_xgb.best_estimator_

y_pred_xgb_tuned = best_xgb.predict(
    X_test_xgb
)

print("Best Parameters:")
print(grid_xgb.best_params_)

print("\nTuned XGBoost:")
print(
    classification_report(
        y_test_xgb,
        y_pred_xgb_tuned,
        target_names=label_encoder.classes_
    )
)

**Which hyperparameter optimization technique have you used and why?**

I used GridSearchCV with 5-fold cross-validation to evaluate different combinations of XGBoost hyperparameters and identify the best-performing configuration.

**1. Which Evaluation metrics did you consider for a positive business impact and why?**

I considered Accuracy, Precision, Recall, and F1 Score for evaluating the classification models. Accuracy measures the overall correctness of the model. Precision measures how accurately the model identifies a particular content type. Recall measures how many actual titles of a particular type are correctly identified. F1 Score provides a balance between Precision and Recall. Therefore, F1 Score along with Accuracy was considered important for selecting the final model.

**2. Which ML model did you choose from the above created models as your final prediction model and why?**

In [ ]:
# Compare all three models

model_comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Random Forest',
        'XGBoost'
    ],
    'Accuracy': [
        accuracy_lr,
        accuracy_rf,
        accuracy_xgb
    ],
    'F1 Score': [
        f1_score(
            y_test,
            y_pred_lr,
            pos_label='SHOW'
        ),
        f1_score(
            y_test,
            y_pred_rf,
            pos_label='SHOW'
        ),
        f1_score(
            y_test_xgb,
            y_pred_xgb,
            average='weighted'
        )
    ]
})

print(model_comparison)

**1. Which Evaluation metrics did you consider for a positive business impact and why?**

I considered Accuracy, Precision, Recall, and F1 Score to evaluate the classification models. Accuracy measures the overall percentage of correctly classified Movies and TV Shows. Precision is important because it indicates how accurately the model identifies a particular content type. Recall measures the model's ability to correctly identify all relevant instances of a content type. F1 Score is particularly important because the dataset is imbalanced, with Movies representing approximately 86.25% and TV Shows approximately 13.75%. Therefore, F1 Score provides a better balance between Precision and Recall and helps ensure that the minority TV Show class is not ignored. These metrics help Amazon Prime build a reliable automated content classification system.

**2. Which ML model did you choose from the above created models as your final prediction model and why?**

I selected XGBoost as the final prediction model. The three models achieved the following results:

Logistic Regression: 94.93% Accuracy, 84.18% F1 Score
Random Forest: 97.47% Accuracy, 90.81% F1 Score
XGBoost: 97.32% Accuracy, 97.32% F1 Score

Although Random Forest achieved slightly higher accuracy than XGBoost, XGBoost achieved the highest F1 Score of 97.32%. Since the dataset is imbalanced, F1 Score is more informative than accuracy alone. Therefore, XGBoost was selected as the final model because it provides a strong balance between correctly identifying Movies and TV Shows.

**3. Explain the model which you have used and the feature importance using any model explainability tool?**

XGBoost (Extreme Gradient Boosting) was selected as the final classification model. It is an ensemble learning algorithm that builds multiple decision trees sequentially, where each new tree attempts to correct the errors made by previous trees. This enables XGBoost to capture complex and nonlinear relationships between the input features and the target variable.

For model interpretation, XGBoost feature importance was used to identify the features that contributed most to the prediction of Movie versus TV Show. The importance scores indicate how much each feature contributed to the model's decision-making process. Features with higher importance have a greater influence on the prediction.

In this project, features such as runtime, IMDb score, IMDb votes, TMDB popularity, TMDB score, and age certification were used for prediction. The seasons feature was excluded because it completely separates the two classes in this dataset (seasons = 0 for all Movies and seasons > 0 for all TV Shows), which would make the prediction trivial.

Feature importance provides useful business insight because it helps Amazon Prime understand which catalog attributes are most useful for automatically classifying and organizing its content.

# **Conclusion**

This project analyzed the Amazon Prime catalog to understand patterns in Movies and TV Shows based on content type, release year, genres, ratings, runtime, countries, and other metadata. Extensive exploratory data analysis was performed using univariate, bivariate, and multivariate visualizations to identify meaningful relationships and trends.

The dataset was cleaned by handling missing values, removing duplicates, standardizing categorical data, and preprocessing textual descriptions. Feature engineering was performed to generate useful numerical and textual features. The seasons feature was excluded from the final ML model because it directly revealed the content type.

Three classification models—Logistic Regression, Random Forest, and XGBoost—were evaluated. XGBoost achieved the highest F1 Score of 97.32%, making it the final selected model. The results demonstrate that machine learning can effectively classify Amazon Prime catalog content using available metadata.

Overall, the analysis provides useful insights for content organization, catalog management, regional content strategy, genre planning, and automated classification, which can support data-driven decision-making for Amazon Prime.